## 1. Import libraries

In [2]:
# ============================================================
# NOTEBOOK 03
# RESAMPLING AND 3D PATCH EXTRACTION
# ============================================================

import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import zoom

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ============================================================
# PREPROCESSED DATASET PATH
# ============================================================

PREPROCESSED_ROOT = "/kaggle/input/YOUR-PREPROCESSED-DATASET-NAME"

In [ ]:
# ============================================================
# INPUT FILES
# ============================================================

CT_FILE = os.path.join(
    PREPROCESSED_ROOT,
    "RibFrac1_ct_normalized.npy"
)

LABEL_FILE = os.path.join(
    PREPROCESSED_ROOT,
    "RibFrac1_label.npy"
)

print("CT file:")
print(CT_FILE)

print("\nLabel file:")
print(LABEL_FILE)

In [ ]:
# ============================================================
# VERIFY INPUT FILES
# ============================================================

if not os.path.isfile(CT_FILE):
    raise FileNotFoundError(
        f"Preprocessed CT file not found:\n{CT_FILE}"
    )

if not os.path.isfile(LABEL_FILE):
    raise FileNotFoundError(
        f"Label file not found:\n{LABEL_FILE}"
    )

print("Preprocessed CT found.")
print("Label file found.")

In [ ]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

ct_volume = np.load(CT_FILE)

label_volume = np.load(LABEL_FILE)

print("CT shape:", ct_volume.shape)
print("Label shape:", label_volume.shape)

print("CT dtype:", ct_volume.dtype)
print("Label dtype:", label_volume.dtype)

In [ ]:
# ============================================================
# INPUT VALIDATION
# ============================================================

assert ct_volume.ndim == 3
assert label_volume.ndim == 3

assert ct_volume.shape == label_volume.shape

assert np.isfinite(ct_volume).all()
assert np.isfinite(label_volume).all()

assert np.min(ct_volume) >= 0.0
assert np.max(ct_volume) <= 1.0

print("Input validation successful.")

In [ ]:
# ============================================================
# CREATE BINARY FRACTURE MASK
# ============================================================

fracture_mask = (label_volume > 0).astype(np.uint8)

print("Original labels:")
print(np.unique(label_volume))

print("\nBinary mask labels:")
print(np.unique(fracture_mask))

In [ ]:
# ============================================================
# FRACTURE VOXEL STATISTICS
# ============================================================

total_voxels = fracture_mask.size

background_voxels = np.sum(fracture_mask == 0)

fracture_voxels = np.sum(fracture_mask == 1)

print("Total voxels:", total_voxels)
print("Background voxels:", background_voxels)
print("Fracture voxels:", fracture_voxels)

print(
    "Fracture voxel percentage:",
    100 * fracture_voxels / total_voxels
)

In [ ]:
# ============================================================
# FRACTURE VOXEL STATISTICS
# ============================================================

total_voxels = fracture_mask.size

background_voxels = np.sum(fracture_mask == 0)

fracture_voxels = np.sum(fracture_mask == 1)

print("Total voxels:", total_voxels)
print("Background voxels:", background_voxels)
print("Fracture voxels:", fracture_voxels)

print(
    "Fracture voxel percentage:",
    100 * fracture_voxels / total_voxels
)

In [ ]:
# ============================================================
# READ ORIGINAL VOXEL SPACING
# ============================================================

original_ct_img = nib.load(ORIGINAL_CT_PATH)

original_spacing = np.array(
    original_ct_img.header.get_zooms()[:3],
    dtype=np.float32
)

print("Original voxel spacing:")
print(original_spacing)

In [ ]:
# ============================================================
# TARGET ISOTROPIC SPACING
# ============================================================

TARGET_SPACING = np.array(
    [1.5, 1.5, 1.5],
    dtype=np.float32
)

print("Original spacing:", original_spacing)
print("Target spacing:", TARGET_SPACING)

In [ ]:
# ============================================================
# RESAMPLING FACTORS
# ============================================================

resize_factor = original_spacing / TARGET_SPACING

print("Resampling factors:")
print(resize_factor)

In [ ]:
# ============================================================
# NEW VOLUME DIMENSIONS
# ============================================================

new_shape = np.round(
    np.array(ct_volume.shape) * resize_factor
).astype(int)

print("Original shape:", ct_volume.shape)
print("New resampled shape:", tuple(new_shape))

In [ ]:
# ============================================================
# RESAMPLE CT
# ============================================================

ct_resampled = zoom(
    ct_volume,
    resize_factor,
    order=1
).astype(np.float32)

print("Resampled CT shape:", ct_resampled.shape)

In [ ]:
# ============================================================
# RESAMPLE FRACTURE MASK
# ============================================================

mask_resampled = zoom(
    fracture_mask,
    resize_factor,
    order=0
).astype(np.uint8)

print("Resampled mask shape:", mask_resampled.shape)
print("Mask labels:", np.unique(mask_resampled))

In [3]:
# ============================================================
# RESAMPLE FRACTURE MASK
# ============================================================

mask_resampled = zoom(
    fracture_mask,
    resize_factor,
    order=0
).astype(np.uint8)

print("Resampled mask shape:", mask_resampled.shape)
print("Mask labels:", np.unique(mask_resampled))

NameError: name 'zoom' is not defined

In [ ]:
# ============================================================
# RESAMPLING VALIDATION
# ============================================================

assert ct_resampled.shape == mask_resampled.shape

assert np.isfinite(ct_resampled).all()

assert np.min(ct_resampled) >= 0.0
assert np.max(ct_resampled) <= 1.0

assert set(np.unique(mask_resampled)).issubset({0, 1})

print("Resampling validation successful.")

print("CT:", ct_resampled.shape)
print("Mask:", mask_resampled.shape)

In [ ]:
# ============================================================
# VISUALIZE RESAMPLED CT
# ============================================================

middle_slice = ct_resampled.shape[2] // 2

plt.figure(figsize=(7, 7))

plt.imshow(
    ct_resampled[:, :, middle_slice],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    f"Resampled CT - Slice {middle_slice}"
)

plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# VISUALIZE RESAMPLED MASK
# ============================================================

plt.figure(figsize=(7, 7))

plt.imshow(
    mask_resampled[:, :, middle_slice]
)

plt.title(
    f"Resampled Fracture Mask - Slice {middle_slice}"
)

plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CT + FRACTURE MASK
# ============================================================

plt.figure(figsize=(8, 8))

plt.imshow(
    ct_resampled[:, :, middle_slice],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.imshow(
    mask_resampled[:, :, middle_slice],
    alpha=0.4
)

plt.title(
    f"Resampled CT + Fracture Mask\n"
    f"Slice {middle_slice}"
)

plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# 3D PATCH CONFIGURATION
# ============================================================

PATCH_SIZE = (64, 64, 64)

print("Patch size:", PATCH_SIZE)

In [ ]:
# ============================================================
# 3D PATCH CONFIGURATION
# ============================================================

PATCH_SIZE = (64, 64, 64)

print("Patch size:", PATCH_SIZE)

In [ ]:
# ============================================================
# FIND FRACTURE VOXELS
# ============================================================

fracture_coordinates = np.argwhere(
    mask_resampled == 1
)

print(
    "Number of fracture voxels:",
    len(fracture_coordinates)
)

In [ ]:
# ============================================================
# PATCH EXTRACTION FUNCTION
# ============================================================

def extract_patch(volume, center, patch_size):
    """
    Extract a 3D patch centered at a specified voxel.

    Parameters
    ----------
    volume : np.ndarray
        3D input volume.

    center : tuple/list
        Center coordinates (x, y, z).

    patch_size : tuple
        Patch dimensions (px, py, pz).

    Returns
    -------
    patch : np.ndarray
        Extracted 3D patch.
    """

    px, py, pz = patch_size

    cx, cy, cz = center

    x_start = cx - px // 2
    y_start = cy - py // 2
    z_start = cz - pz // 2

    x_end = x_start + px
    y_end = y_start + py
    z_end = z_start + pz

    # Check boundaries
    if (
        x_start < 0 or
        y_start < 0 or
        z_start < 0 or
        x_end > volume.shape[0] or
        y_end > volume.shape[1] or
        z_end > volume.shape[2]
    ):
        return None

    patch = volume[
        x_start:x_end,
        y_start:y_end,
        z_start:z_end
    ]

    return patch

In [ ]:
# ============================================================
# PATCH SAMPLING CONFIGURATION
# ============================================================

NUM_POSITIVE_PATCHES = 128
NUM_NEGATIVE_PATCHES = 128

RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

print("Positive patches:", NUM_POSITIVE_PATCHES)
print("Negative patches:", NUM_NEGATIVE_PATCHES)
print("Total patches:", NUM_POSITIVE_PATCHES + NUM_NEGATIVE_PATCHES)

In [ ]:
# ============================================================
# EXTRACT POSITIVE PATCHES
# ============================================================

positive_ct_patches = []
positive_mask_patches = []

attempts = 0
max_attempts = NUM_POSITIVE_PATCHES * 20

while (
    len(positive_ct_patches) < NUM_POSITIVE_PATCHES
    and attempts < max_attempts
):

    attempts += 1

    index = rng.integers(
        0,
        len(fracture_coordinates)
    )

    center = fracture_coordinates[index]

    ct_patch = extract_patch(
        ct_resampled,
        center,
        PATCH_SIZE
    )

    mask_patch = extract_patch(
        mask_resampled,
        center,
        PATCH_SIZE
    )

    if ct_patch is None or mask_patch is None:
        continue

    # Make sure the patch actually contains fracture
    if np.sum(mask_patch) == 0:
        continue

    positive_ct_patches.append(ct_patch)
    positive_mask_patches.append(mask_patch)

print(
    "Positive patches extracted:",
    len(positive_ct_patches)
)

In [ ]:
# ============================================================
# EXTRACT NEGATIVE PATCHES
# ============================================================

negative_ct_patches = []
negative_mask_patches = []

attempts = 0
max_attempts = NUM_NEGATIVE_PATCHES * 50

volume_shape = ct_resampled.shape

while (
    len(negative_ct_patches) < NUM_NEGATIVE_PATCHES
    and attempts < max_attempts
):

    attempts += 1

    center = [
        rng.integers(PATCH_SIZE[0] // 2,
                     volume_shape[0] - PATCH_SIZE[0] // 2),

        rng.integers(PATCH_SIZE[1] // 2,
                     volume_shape[1] - PATCH_SIZE[1] // 2),

        rng.integers(PATCH_SIZE[2] // 2,
                     volume_shape[2] - PATCH_SIZE[2] // 2)
    ]

    ct_patch = extract_patch(
        ct_resampled,
        center,
        PATCH_SIZE
    )

    mask_patch = extract_patch(
        mask_resampled,
        center,
        PATCH_SIZE
    )

    if ct_patch is None or mask_patch is None:
        continue

    # Negative patch must contain no fracture
    if np.sum(mask_patch) != 0:
        continue

    negative_ct_patches.append(ct_patch)
    negative_mask_patches.append(mask_patch)

print(
    "Negative patches extracted:",
    len(negative_ct_patches)
)

In [ ]:
# ============================================================
# CONVERT PATCHES TO ARRAYS
# ============================================================

X_positive = np.asarray(
    positive_ct_patches,
    dtype=np.float32
)

Y_positive = np.asarray(
    positive_mask_patches,
    dtype=np.float32
)

X_negative = np.asarray(
    negative_ct_patches,
    dtype=np.float32
)

Y_negative = np.asarray(
    negative_mask_patches,
    dtype=np.float32
)

print("Positive CT patches:", X_positive.shape)
print("Positive masks:", Y_positive.shape)

print("Negative CT patches:", X_negative.shape)
print("Negative masks:", Y_negative.shape)

In [ ]:
# ============================================================
# CONVERT PATCHES TO ARRAYS
# ============================================================

X_positive = np.asarray(
    positive_ct_patches,
    dtype=np.float32
)

Y_positive = np.asarray(
    positive_mask_patches,
    dtype=np.float32
)

X_negative = np.asarray(
    negative_ct_patches,
    dtype=np.float32
)

Y_negative = np.asarray(
    negative_mask_patches,
    dtype=np.float32
)

print("Positive CT patches:", X_positive.shape)
print("Positive masks:", Y_positive.shape)

print("Negative CT patches:", X_negative.shape)
print("Negative masks:", Y_negative.shape)

In [ ]:
# ============================================================
# CLASSIFICATION LABELS
# ============================================================

classification_positive = np.ones(
    len(X_positive),
    dtype=np.float32
)

classification_negative = np.zeros(
    len(X_negative),
    dtype=np.float32
)

Y_class = np.concatenate(
    [
        classification_positive,
        classification_negative
    ],
    axis=0
)

print("Classification label shape:", Y_class.shape)

print(
    "Class 0:",
    np.sum(Y_class == 0)
)

print(
    "Class 1:",
    np.sum(Y_class == 1)
)

In [ ]:
# ============================================================
# ADD CHANNEL DIMENSION
# ============================================================

X = X[..., np.newaxis]

Y_seg = Y_seg[..., np.newaxis]

print("X shape:", X.shape)
print("Y segmentation shape:", Y_seg.shape)
print("Y classification shape:", Y_class.shape)

In [ ]:
# ============================================================
# SHUFFLE PATCHES
# ============================================================

indices = rng.permutation(len(X))

X = X[indices]
Y_seg = Y_seg[indices]
Y_class = Y_class[indices]

print("Patch order randomized.")

In [ ]:
# ============================================================
# CHECK CLASS BALANCE
# ============================================================

num_class_0 = np.sum(Y_class == 0)
num_class_1 = np.sum(Y_class == 1)

print("Class 0 - Background:", num_class_0)
print("Class 1 - Fracture:", num_class_1)

print(
    "Class 0 percentage:",
    100 * num_class_0 / len(Y_class)
)

print(
    "Class 1 percentage:",
    100 * num_class_1 / len(Y_class)
)

In [ ]:
# ============================================================
# SEGMENTATION PATCH STATISTICS
# ============================================================

positive_patch_count = np.sum(
    np.sum(Y_seg, axis=(1, 2, 3, 4)) > 0
)

negative_patch_count = np.sum(
    np.sum(Y_seg, axis=(1, 2, 3, 4)) == 0
)

print(
    "Patches containing fracture:",
    positive_patch_count
)

print(
    "Background-only patches:",
    negative_patch_count
)

In [ ]:
# ============================================================
# SEGMENTATION PATCH STATISTICS
# ============================================================

positive_patch_count = np.sum(
    np.sum(Y_seg, axis=(1, 2, 3, 4)) > 0
)

negative_patch_count = np.sum(
    np.sum(Y_seg, axis=(1, 2, 3, 4)) == 0
)

print(
    "Patches containing fracture:",
    positive_patch_count
)

print(
    "Background-only patches:",
    negative_patch_count
)

In [ ]:
# ============================================================
# VISUALIZE A POSITIVE PATCH
# ============================================================

positive_indices = np.where(Y_class == 1)[0]

sample_index = positive_indices[0]

sample_ct = X[sample_index, ..., 0]
sample_mask = Y_seg[sample_index, ..., 0]

slice_index = PATCH_SIZE[2] // 2

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)

plt.imshow(
    sample_ct[:, :, slice_index],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title("Positive CT Patch")
plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(
    sample_ct[:, :, slice_index],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.imshow(
    sample_mask[:, :, slice_index],
    alpha=0.4
)

plt.title("CT + Fracture Mask")
plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# SAVE PATCH DATA
# ============================================================

OUTPUT_DIR = "/kaggle/working/patch_dataset"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

X_PATH = os.path.join(
    OUTPUT_DIR,
    "X_patches.npy"
)

Y_SEG_PATH = os.path.join(
    OUTPUT_DIR,
    "Y_segmentation.npy"
)

Y_CLASS_PATH = os.path.join(
    OUTPUT_DIR,
    "Y_classification.npy"
)

np.save(X_PATH, X)
np.save(Y_SEG_PATH, Y_seg)
np.save(Y_CLASS_PATH, Y_class)

print("Patch dataset saved.")

print(X_PATH)
print(Y_SEG_PATH)
print(Y_CLASS_PATH)

In [ ]:
# ============================================================
# SAVE PATCH CONFIGURATION
# ============================================================

patch_metadata = {
    "original_shape": tuple(ct_volume.shape),
    "resampled_shape": tuple(ct_resampled.shape),
    "original_spacing": original_spacing.tolist(),
    "target_spacing": TARGET_SPACING.tolist(),
    "patch_size": PATCH_SIZE,
    "number_positive_patches": int(np.sum(Y_class == 1)),
    "number_negative_patches": int(np.sum(Y_class == 0)),
    "classification_classes": {
        "0": "background/no fracture",
        "1": "fracture present"
    },
    "segmentation_classes": {
        "0": "background",
        "1": "fracture"
    }
}

metadata_path = os.path.join(
    OUTPUT_DIR,
    "patch_metadata.npy"
)

np.save(
    metadata_path,
    patch_metadata,
    allow_pickle=True
)

print("Patch metadata saved.")
print(metadata_path)

In [ ]:
# ============================================================
# FINAL NOTEBOOK 03 VALIDATION
# ============================================================

assert X.ndim == 5
assert Y_seg.ndim == 5
assert Y_class.ndim == 1

assert X.shape[1:] == (
    64, 64, 64, 1
)

assert Y_seg.shape[1:] == (
    64, 64, 64, 1
)

assert X.shape[0] == Y_seg.shape[0]
assert X.shape[0] == Y_class.shape[0]

assert np.isfinite(X).all()
assert np.isfinite(Y_seg).all()
assert np.isfinite(Y_class).all()

assert set(np.unique(Y_seg)).issubset({0, 1})
assert set(np.unique(Y_class)).issubset({0, 1})

print("==============================================")
print("NOTEBOOK 03 - PATCH EXTRACTION SUCCESSFUL")
print("==============================================")

print("CT patch shape:", X.shape)
print("Segmentation target:", Y_seg.shape)
print("Classification target:", Y_class.shape)

print()
print("Fracture patches:", np.sum(Y_class == 1))
print("Background patches:", np.sum(Y_class == 0))

print()
print("Patch size:", PATCH_SIZE)
print("Target spacing:", TARGET_SPACING)

print("==============================================")